In [0]:
from pyspark.sql.functions import *

silver_orders = spark.table("workspace.silver.silver_orders")
silver_customers = spark.table("workspace.silver.silver_customers")
silver_products = spark.table("workspace.silver.silver_products")
silver_order_items = spark.table("workspace.silver.silver_order_items")
silver_exchange_rates = spark.table("workspace.silver.silver_exchange_rates")

In [0]:
# dim_customers
dim_customers = silver_customers.select(
    "customer_id",
    "customer_name",
    "email",
    "country_code"
).dropDuplicates()

# dim_products
dim_products = silver_products.select(
    "product_id",
    "product_name",
    "category",
    "price",
    "country_code"
).dropDuplicates()

# dim_status
dim_status = silver_orders.select("order_status") \
    .distinct() \
    .withColumn("status_id", monotonically_increasing_id())

# dim_date
dim_date = silver_orders.select(
    col("order_date").alias("date")
).dropDuplicates() \
.withColumn("year", year("date")) \
.withColumn("month", month("date")) \
.withColumn("day", dayofmonth("date")) \
.withColumn("quarter", quarter("date"))

In [0]:
silver_orders = spark.table("workspace.silver.silver_orders")

dim_date = silver_orders.select(
    to_date(col("order_date"), 'dd-MM-yyyy').alias("date")
).dropDuplicates() \
.withColumn("year", year("date")) \
.withColumn("month", month("date")) \
.withColumn("day", dayofmonth("date"))


In [0]:
from pyspark.sql.functions import *

# FACT TABLE with currency conversion
fact_orders = silver_order_items \
    .join(silver_orders, "order_id", "left") \
    .join(dim_products, "product_id", "left") \
    .join(dim_customers, "customer_id", "left") \
    .join(dim_status, silver_orders.order_status == dim_status.order_status, "left") \
    .withColumn(
        "revenue_base",
        round(col("quantity") * col("unit_price"),2)
    ) \
    .select(
        col("order_id"),
        col("customer_id"),
        col("product_id"),
        col("order_date"),
        col("quantity"),
        col("unit_price"),
        col("revenue_base"),
        col("status_id"),
        silver_products.country_code,
        silver_orders.channel
    )
    # .join(silver_exchange_rates, "currency_code", "left") \
        #  * col("exchange_rate_to_usd")

display(fact_orders)

In [0]:
dim_customers.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.dim_customers")
dim_products.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.dim_products")
dim_status.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.dim_status")
dim_date.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.dim_date")

fact_orders.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.fact_orders")

In [0]:
# Total Revenue
kpi_total_revenue = fact_orders.agg(
    sum("revenue_base").alias("total_revenue")
)
display(kpi_total_revenue)


In [0]:

# Revenue by country
kpi_revenue_country = fact_orders.groupBy("country_code") \
    .agg(round(sum("revenue_base"),2).alias("revenue"))

display(kpi_revenue_country)

In [0]:
# Revenue by channel 
kpi_revenue_channel = fact_orders.groupBy("channel") \
    .agg(round(sum("revenue_base"),2).alias("revenue"))

display(kpi_revenue_channel)

In [0]:
# Completed order count
kpi_completed_orders = fact_orders \
    .join(dim_status, "status_id") \
    .filter(col("order_status") == "completed") \
    .agg(countDistinct("order_id").alias("completed_orders"))

display(kpi_completed_orders)


In [0]:
# Completed order rate
total_orders = fact_orders.select(countDistinct("order_id")).collect()[0][0]

completed_order_rate = kpi_completed_orders.collect()[0][0] /total_orders
display(total_orders)

In [0]:
# AOV
kpi_aov = fact_orders.groupBy("order_id") \
    .agg(round(sum("revenue_base"),2).alias("order_value")) \
    .agg(round(avg("order_value"),2).alias("AOV"))
display(kpi_aov)


In [0]:
# Top 5 products by revenue
kpi_top_products = fact_orders \
    .groupBy("product_id") \
    .agg(round(sum("revenue_base"),2).alias("revenue")) \
    .orderBy(desc("revenue")) \
    .limit(5)
display(kpi_top_products)


In [0]:
# active customers
kpi_active_customers = fact_orders \
    .select(countDistinct("customer_id").alias("active_customers"))
display(kpi_active_customers)


In [0]:
# customer acquistion 
kpi_customer_acquisition = dim_customers \
    .join(silver_orders, "customer_id") \
    .withColumn("month", month("order_date")) \
    .groupBy("month") \
    .agg(countDistinct("customer_id").alias("new_customers"))
display(kpi_customer_acquisition)


In [0]:
# Data quality score
kpi_data_quality = silver_orders.agg(
    (sum(col("is_valid")) / count("*")).alias("data_quality_score")
)
display(kpi_data_quality)